In [1]:
# 04 Error analysis
#Where does Logistic Regression go wrong, with and without MMSE? (RQ1 and RQ2)

In [2]:
import warnings
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")

df = pd.read_csv("../data/oasis_longitudinal.csv")
df["label"] = (df["CDR"] > 0).astype(int)
df["Sex"] = (df["M/F"] == "M").astype(int)

y = df["label"]
groups = df["Subject ID"]

feature_sets = {
    "B (no MMSE)": ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV"],
    "A (with MMSE)": ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV", "MMSE"],
}

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

In [3]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

for exp_name, feats in feature_sets.items():
    pred = cross_val_predict(model, df[feats], y, groups=groups, cv=cv)
    cm = confusion_matrix(y, pred)
    table = pd.DataFrame(
        cm,
        index=["Actual: no dementia signs", "Actual: dementia signs"],
        columns=["Predicted: no signs", "Predicted: signs"],
    )
    print(exp_name)
    print(table)
    print()

B (no MMSE)
                           Predicted: no signs  Predicted: signs
Actual: no dementia signs                  156                50
Actual: dementia signs                      61               106

A (with MMSE)
                           Predicted: no signs  Predicted: signs
Actual: no dementia signs                  182                24
Actual: dementia signs                      46               121



In [4]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

for exp_name, feats in feature_sets.items():
    pred = cross_val_predict(model, df[feats], y, groups=groups, cv=cv)

    check = df[["Subject ID", "CDR"]].copy()
    check["pred"] = pred
    sick = check[y == 1]   # only visits with dementia signs

    summary = sick.groupby("CDR").agg(total=("pred", "size"), caught=("pred", "sum"))
    summary["missed"] = summary["total"] - summary["caught"]
    summary["miss_rate"] = (summary["missed"] / summary["total"]).round(2)

    print(exp_name)
    print(summary)
    print()

B (no MMSE)
     total  caught  missed  miss_rate
CDR                                  
0.5    123      73      50       0.41
1.0     41      32       9       0.22
2.0      3       1       2       0.67

A (with MMSE)
     total  caught  missed  miss_rate
CDR                                  
0.5    123      78      45       0.37
1.0     41      40       1       0.02
2.0      3       3       0       0.00

